# Day 22 — Capstone: **Atlas v1**

Assemble everything from Days 15–21 into one small but complete agent:

- the **ReAct loop** in a reusable `Atlas` class (Day 16, 19),
- a **resilient** model call (Day 20),
- three **tools** — calculator, clock, word_count (Day 13, 17),
- optional **reflection** before finishing (Day 21).

Your task: wire the reflection step into `finish`. The solution shows the full agent.
This is *Atlas v1* — the same shape scales all the way to the Phase 9 capstone.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run the cell. Stuck? The solution is below — but try first.

In [ ]:
import json
from shared.llm import chat
from shared.tools import calculator, clock, word_count
from shared.reliability import resilient

@resilient(attempts=3, time_budget=30.0)
def robust_chat(messages):
    return chat(messages, temperature=0)

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

class Atlas:
    def __init__(self, tools, max_steps=8):
        self.tools = tools
        self.max_steps = max_steps
        names = "|".join(list(tools) + ["finish"])
        self.system = (
            "You are Atlas. Reason then act. Reply ONLY with JSON "
            '{"thought":"...","action":"' + names + '","action_input":"..."}. '
            "Tools take a single string. Use 'finish' for the final answer."
        )

    def run(self, goal, reflect=True, verbose=True):
        messages = [{"role": "system", "content": self.system},
                    {"role": "user", "content": goal}]
        for step in range(1, self.max_steps + 1):
            obj = parse_json(robust_chat(messages))
            if verbose:
                print(f"[{step}] 💭 {obj.get('thought', '')}")
            if obj.get("action") == "finish":
                answer = obj.get("action_input", "")
                # TODO: if reflect, critique `answer` vs `goal`; if not 'OK', revise ONCE and return it.
                return answer
            tool = self.tools.get(obj.get("action"))
            obs = tool(obj.get("action_input", "")) if tool else f"unknown tool: {obj.get('action')}"
            if verbose:
                print(f"     🔧 {obj.get('action')} -> {obs}")
            messages.append({"role": "assistant", "content": json.dumps(obj)})
            messages.append({"role": "user", "content": f"Observation: {obs}"})
        return "stopped (hit max_steps)"

# atlas = Atlas({"calculator": calculator, "clock": clock, "word_count": word_count})
# print(atlas.run("Compute 7 * 6, then count the words in 'agents act in a loop'."))

## 🔒 Solution

One correct way to do it. Compare it with your version.

In [ ]:
import json
from shared.llm import chat
from shared.tools import calculator, clock, word_count
from shared.reliability import resilient

@resilient(attempts=3, time_budget=30.0)
def robust_chat(messages):
    return chat(messages, temperature=0)

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

def reflect_once(goal, answer):
    c = chat([
        {"role": "system", "content": "Strict reviewer. Reply 'OK' if the answer fully addresses the goal, else a short fix instruction."},
        {"role": "user", "content": f"Goal: {goal}\nAnswer: {answer}"},
    ], temperature=0)
    if c.strip().upper().startswith("OK"):
        return answer
    return chat([
        {"role": "system", "content": "Revise to satisfy the reviewer. Return only the improved answer."},
        {"role": "user", "content": f"Goal: {goal}\nDraft: {answer}\nReviewer: {c}"},
    ], temperature=0)

class Atlas:
    def __init__(self, tools, max_steps=8):
        self.tools = tools
        self.max_steps = max_steps
        names = "|".join(list(tools) + ["finish"])
        self.system = (
            "You are Atlas. Reason then act. Reply ONLY with JSON "
            '{"thought":"...","action":"' + names + '","action_input":"..."}. '
            "Tools take a single string. Use 'finish' for the final answer."
        )

    def run(self, goal, reflect=True, verbose=True):
        messages = [{"role": "system", "content": self.system},
                    {"role": "user", "content": goal}]
        for step in range(1, self.max_steps + 1):
            obj = parse_json(robust_chat(messages))
            if verbose:
                print(f"[{step}] 💭 {obj.get('thought', '')}")
            if obj.get("action") == "finish":
                answer = obj.get("action_input", "")
                return reflect_once(goal, answer) if reflect else answer
            tool = self.tools.get(obj.get("action"))
            obs = tool(obj.get("action_input", "")) if tool else f"unknown tool: {obj.get('action')}"
            if verbose:
                print(f"     🔧 {obj.get('action')} -> {obs}")
            messages.append({"role": "assistant", "content": json.dumps(obj)})
            messages.append({"role": "user", "content": f"Observation: {obs}"})
        return "stopped (hit max_steps)"

atlas = Atlas({"calculator": calculator, "clock": clock, "word_count": word_count})
print(atlas.run("Compute 7 * 6, then count the words in 'agents act in a loop'."))